In [1]:
from pyspark.sql.functions import col, trim, upper, when, lit, current_timestamp, date_format
from pyspark.sql.types import DoubleType, IntegerType,StringType

StatementMeta(sparkpoolbank, 83, 2, Finished, Available, Finished, False)

In [2]:
bronze_path = "abfss://bronze@stbankingretail.dfs.core.windows.net/Loan_Details/*.csv"

df_bronze = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(bronze_path)

# df_bronze.show()

StatementMeta(sparkpoolbank, 83, 3, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    date_format,
    lit,
    trim,
    upper,
    when,
)
from pyspark.sql.types import DoubleType, IntegerType, StringType

# 0. Set Spark Configurations for INT96 Parquet Timestamps
spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")
spark.conf.set("spark.sql.parquet.writelegacyformat", "true")

# 1. Transformations
df_loan_clean = (
    df_bronze.withColumn("LoanID", trim(col("Loan_ID")))
    .withColumn("CustomerID", trim(col("Customer_ID")))
    .withColumn("LoanAmount", col("Loan_Amount").cast(DoubleType()))
    .withColumn("EMI", col("EMI").cast(DoubleType()))
    .withColumn("PendingEMIs", col("Pending_EMIs").cast(IntegerType()))
    .withColumn(
        "DefaultStatus",
        when(upper(trim(col("Default_Status"))).isin("Y", "YES"), lit("Yes"))
        .when(upper(trim(col("Default_Status"))).isin("N", "NO"), lit("No"))
        .otherwise(lit("Unknown")),
    )
    # Explicitly cast IngestionDate to StringType
    .withColumn(
        "IngestionDate",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss").cast(
            StringType()
        ),
    )
)

# 2. Select final columns
df_loan_final = df_loan_clean.select(
    "LoanID",
    "CustomerID",
    "LoanAmount",
    "EMI",
    "PendingEMIs",
    "DefaultStatus",
    "IngestionDate",
)

# 3. Save as Parquet
silver_path = (
    "abfss://silver@stbankingretail.dfs.core.windows.net/Loan_Details"
)

df_loan_final.coalesce(1).write.format("parquet").option(
    "parquet.writer.version", "v1"
).mode("overwrite").option("overwriteSchema", "true").save(silver_path)

StatementMeta(sparkpoolbank, 83, 4, Finished, Available, Finished, False)

In [9]:
# df_loan_final.count()

StatementMeta(sparkpoolbank, 16, 9, Finished, Available, Finished, False)

300